In [2]:
import pandas as pd
import numpy as np
import os

# CONFIGURATION
PRIMARY_WAVE = "o"        # Wave 15
BACKUP_WAVES = ["n", "m", "l", "k"] # Waves 14 and 13
BASE_PICKLE_DIR = "../data/ukhls/pickles"
OUTPUT_FILE = os.path.join(BASE_PICKLE_DIR, f"{PRIMARY_WAVE}_indresp_backfilled.pkl")

print(f"Targeting Wave {PRIMARY_WAVE} with backups from {BACKUP_WAVES}")

def safe_numeric(series):
    numeric = pd.to_numeric(series, errors='coerce')
    if pd.api.types.is_numeric_dtype(numeric):
        numeric = numeric.replace([np.inf, -np.inf], np.nan)
    return numeric

def build_expanded_master():
    primary_file = os.path.join(BASE_PICKLE_DIR, f"{PRIMARY_WAVE}_indresp_optimized.pkl")
    
    if not os.path.exists(primary_file):
        raise FileNotFoundError(f"Could not find primary wave file: {primary_file}")

    # 1. Load Primary Wave
    print(f"Loading Primary Wave ({PRIMARY_WAVE})...")
    df_master = pd.read_pickle(primary_file).set_index('pidp')
    
    # 2. Iteratively Backfill
    for i, wave in enumerate(BACKUP_WAVES):
        years_to_add = i + 1
        backup_file = os.path.join(BASE_PICKLE_DIR, f"{wave}_indresp_optimized.pkl")
        
        if not os.path.exists(backup_file):
            print(f"Warning: {backup_file} not found. Skipping wave {wave}.")
            continue
            
        print(f"Processing Wave {wave} (Age offset: +{years_to_add})...")
        df_backup = pd.read_pickle(backup_file).set_index('pidp')
        
        # Align column names (e.g., n_age_dv -> o_age_dv)
        df_backup.columns = df_backup.columns.str.replace(f"{wave}_", f"{PRIMARY_WAVE}_")
        
        # A. Pull forward entire missing respondents
        new_ids = df_backup.index.difference(df_master.index)
        if not new_ids.empty:
            df_new = df_backup.loc[new_ids].copy()
            # Age increment logic
            age_col = f"{PRIMARY_WAVE}_age_dv"
            if age_col in df_new.columns:
                age_numeric = safe_numeric(df_new[age_col])
                df_new[age_col] = age_numeric + years_to_add
            
            df_master = pd.concat([df_master, df_new])
            print(f"   -> Added {len(new_ids):,} missing respondents.")

        # B. Patch holes in existing respondents
        # Normalize categorical/int conflicts to object before combine_first
        common_cols = df_master.columns.intersection(df_backup.columns)
        for col in common_cols:
            master_dtype = df_master[col].dtype
            backup_dtype = df_backup[col].dtype
            if isinstance(master_dtype, pd.CategoricalDtype) or isinstance(backup_dtype, pd.CategoricalDtype):
                df_master[col] = df_master[col].astype('object')
                df_backup[col] = df_backup[col].astype('object')

        df_master = df_master.combine_first(df_backup)
        print(f"   -> Patched missing variable values.")

    # 3. Final Optimization
    print("\nOptimizing final dataframe...")
    df_master = df_master.reset_index()
    
    for col in df_master.columns:
        # Ensure IDs are integer-like and NaN-safe
        if 'idp' in col.lower():
            id_numeric = safe_numeric(df_master[col]).fillna(0)
            df_master[col] = id_numeric.astype(np.int64)
        # Category conversion for small-unique-set columns
        elif df_master[col].nunique() < 100 and not pd.api.types.is_float_dtype(df_master[col]):
            df_master[col] = df_master[col].astype('category')
        # Downcast ints/floats with NaN-safe coercion
        elif pd.api.types.is_integer_dtype(df_master[col]):
            int_numeric = safe_numeric(df_master[col])
            df_master[col] = int_numeric.astype('Int64')
        elif pd.api.types.is_float_dtype(df_master[col]):
            float_numeric = safe_numeric(df_master[col])
            df_master[col] = pd.to_numeric(float_numeric, downcast='float')

    # 4. Save as Protocol 5 Pickle (fastest for modern Python)
    df_master.to_pickle(OUTPUT_FILE, protocol=5)
    print(f"DONE. Final Master size: {len(df_master):,} rows.")
    print(f"File saved to: {OUTPUT_FILE}")

build_expanded_master()

Targeting Wave o with backups from ['n', 'm', 'l', 'k']
Loading Primary Wave (o)...
Processing Wave n (Age offset: +1)...
   -> Added 6,503 missing respondents.
   -> Patched missing variable values.
Processing Wave m (Age offset: +2)...
   -> Added 2,450 missing respondents.
   -> Patched missing variable values.
Processing Wave l (Age offset: +3)...
   -> Added 2,460 missing respondents.
   -> Patched missing variable values.
Processing Wave k (Age offset: +4)...
   -> Added 3,092 missing respondents.
   -> Patched missing variable values.

Optimizing final dataframe...
DONE. Final Master size: 47,354 rows.
File saved to: ../data/ukhls/pickles/o_indresp_backfilled.pkl
